In [2]:
function target(x::Float64)::Float64
    return exp(-x)
end

function numerical_d1(f, x::Float64; h::Float64=1e-5)::Float64
    return (f(x + h) - f(x - h)) / (2.0 * h)
end

function numerical_d2(f, x::Float64; h::Float64=1e-4)::Float64
    return (f(x + h) - 2.0 * f(x) + f(x - h)) / (h * h)
end

function svenn(f, x0::Float64; h::Float64=0.1, max_iter::Int=50)
    println("=== Метод Свенна ===")
    println("Начальная точка x0 = $x0,  шаг h = $h")

    f0 = f(x0)
    f1 = f(x0 + h)

    if f1 > f0
        h = -h
        f1 = f(x0 + h)

        if f1 > f0
            a = x0 - abs(h)
            b = x0 + abs(h)
            println("Минимум локализован сразу: [$a, $b]")
            return (a, b)
        end
    end

    x_prev = x0
    x_curr = x0 + h

    for k in 1:max_iter
        step = h * 2.0^k
        x_next = x0 + step
        f_next = f(x_next)
        f_curr = f(x_curr)

        println("  k=$k: x_prev=$(round(x_prev,digits=5)), " *
                "x_curr=$(round(x_curr,digits=5)), " *
                "x_next=$(round(x_next,digits=5)), " *
                "f_next=$(round(f_next,digits=6))")

        if f_next >= f_curr
            a = min(x_prev, x_next)
            b = max(x_prev, x_next)
            println("Найден отрезок: [$a, $b]")
            println("f($a) = $(f(a)),  f($b) = $(f(b))")
            return (a, b)
        end

        x_prev = x_curr
        x_curr = x_next
    end

    a = min(x_prev, x_curr)
    b = max(x_prev, x_curr)
    println("Достигнуто max_iter. Отрезок: [$a, $b]")
    return (a, b)
end

function check_unimodal(f, a::Float64, b::Float64;
                        n::Int=200, tol::Float64=1e-6)
    println("\n=== Проверка унимодальности на [$a, $b] ===")
    h_diff = (b - a) * 1e-5
    h_diff = max(h_diff, 1e-10)

    xs = range(a + h_diff, b - h_diff, length=n)
    prev_d = numerical_d1(f, Float64(xs[1]); h=h_diff)

    for i in 2:n
        x = Float64(xs[i])
        cur_d = numerical_d1(f, x; h=h_diff)
        if cur_d < prev_d - tol
            println("f'(x) убывает в точке x=$(round(x,digits=6)) → функция НЕ унимодальна!")
            return false
        end
        prev_d = cur_d
    end

    println("f'(x) не убывает на всём отрезке → функция унимодальна ✓")
    return true
end

function fibonacci_search(f, a::Float64, b::Float64; eps::Float64=1e-5)
    println("\n=== Метод Фибоначчи ===")
    println("Отрезок: [$a, $b],  точность eps = $eps")

    F = [1.0, 1.0]
    while F[end] < (b - a) / eps
        push!(F, F[end] + F[end-1])
    end
    N = length(F)
    println("Использовано чисел Фибоначчи: N=$N, F[N]=$(F[N])")

    x1 = a + (F[N-2] / F[N]) * (b - a)
    x2 = a + (F[N-1] / F[N]) * (b - a)
    f1 = f(x1)
    f2 = f(x2)

    for k in 1:(N - 3)
        if f1 > f2
            a  = x1
            x1 = x2
            f1 = f2
            x2 = a + (F[N-k-1] / F[N-k]) * (b - a)
            f2 = f(x2)
        else
            b  = x2
            x2 = x1
            f2 = f1
            x1 = a + (F[N-k-2] / F[N-k]) * (b - a)
            f1 = f(x1)
        end
        println("  Шаг k=$k: [$(round(a,digits=6)), $(round(b,digits=6))]," *
                " x1=$(round(x1,digits=6)), x2=$(round(x2,digits=6))")
    end

    x_min = (a + b) / 2.0
    println("\nРезультат:")
    println("  x_min  = $x_min")
    println("  f(x_min) = $(f(x_min))")
    println("  eps    = $eps")
    return x_min
end

function check_minimum(f, x_min::Float64; h::Float64=1e-4)
    println("\n=== Правило дождя (проверка f''(x*)) ===")
    d2 = numerical_d2(f, x_min; h=h)
    println("f''($x_min) ≈ $d2")
    if d2 > 0
        println("f''(x*) > 0  →  точка x* = $x_min является минимумом ✓")
        return true
    elseif d2 < 0
        println("f''(x*) < 0  →  точка x* = $x_min является максимумом ✗")
        return false
    else
        println("f''(x*) ≈ 0  →  точка перегиба или вырожденный случай")
        return false
    end
end

function main()
    println("=" ^ 55)
    println("  Целевая функция: f(x) = e^(-x)")
    println("=" ^ 55)

    x0 = 2.0
    eps = 1e-5

    a, b = svenn(target, x0; h=0.1)

    unimodal = check_unimodal(target, a, b)

    x_min = fibonacci_search(target, a, b; eps=eps)

    check_minimum(target, x_min)

    println("\n" * "=" ^ 55)
    println("ИТОГ:")
    println("  Отрезок (Свенн):    [$a, $b]")
    println("  Унимодальна:        $unimodal")
    println("  x* ≈ $x_min  (точность $eps)")
    println("  f(x*) ≈ $(target(x_min))")
    println("=" ^ 55)
end

main()

  Целевая функция: f(x) = e^(-x)
=== Метод Свенна ===
Начальная точка x0 = 2.0,  шаг h = 0.1
  k=1: x_prev=2.0, x_curr=2.1, x_next=2.2, f_next=0.110803
  k=2: x_prev=2.1, x_curr=2.2, x_next=2.4, f_next=0.090718
  k=3: x_prev=2.2, x_curr=2.4, x_next=2.8, f_next=0.06081
  k=4: x_prev=2.4, x_curr=2.8, x_next=3.6, f_next=0.027324
  k=5: x_prev=2.8, x_curr=3.6, x_next=5.2, f_next=0.005517
  k=6: x_prev=3.6, x_curr=5.2, x_next=8.4, f_next=0.000225
  k=7: x_prev=5.2, x_curr=8.4, x_next=14.8, f_next=0.0
  k=8: x_prev=8.4, x_curr=14.8, x_next=27.6, f_next=0.0
  k=9: x_prev=14.8, x_curr=27.6, x_next=53.2, f_next=0.0
  k=10: x_prev=27.6, x_curr=53.2, x_next=104.4, f_next=0.0
  k=11: x_prev=53.2, x_curr=104.4, x_next=206.8, f_next=0.0
  k=12: x_prev=104.4, x_curr=206.8, x_next=411.6, f_next=0.0
  k=13: x_prev=206.8, x_curr=411.6, x_next=821.2, f_next=0.0
  k=14: x_prev=411.6, x_curr=821.2, x_next=1640.4, f_next=0.0
Найден отрезок: [411.6, 1640.4]
f(411.6) = 1.755461255327414e-179,  f(1640.4) = 0.0